# putEMG — Preprocessing & Feature Extraction

Run in Google Colab. Processes raw `.mat` files into libemg flat-per-rep features.

| Stage | Description | Output |
|-------|-------------|--------|
| **1** | Bandpass → resample → z-score | `UG_per_subject/` |
| **2** | libemg 50-feature flat-per-rep extraction | `feature_gestures/flat_rep/` |

In [ ]:
# Install dependencies (run once per Colab session)
!pip install -q libemg scipy numpy

In [ ]:
from google.colab import drive
import sys

drive.mount('/content/drive')

# Update DRIVE_BASE to match your Google Drive layout
DRIVE_BASE = '/content/drive/MyDrive'
REPO_DIR   = f'{DRIVE_BASE}/putEMG prime/data_preprocessing'
RAW_DIR    = f'{DRIVE_BASE}/putEMG prime/data/NUG_per_subject'
PROC_DIR   = f'{DRIVE_BASE}/putEMG prime/data/UG_per_subject'
FEAT_DIR   = f'{DRIVE_BASE}/putEMG prime/data/feature_gestures/flat_rep'

# Make preprocessing and feature_extraction importable
sys.path.insert(0, REPO_DIR)

---
## Stage 1 — Signal Preprocessing

Bandpass 20–500 Hz → resample to 1500 samples → per-channel z-score normalization.

Input: `NUG_per_subject/` &nbsp; Output: `UG_per_subject/` — key `combinedCell (N_reps, 7)`, each cell `(1500, 24)`.

Already-processed subjects are skipped automatically.

In [ ]:
from preprocessing import batch_process_subjects

batch_process_subjects(
    input_dir      = RAW_DIR,
    output_dir     = PROC_DIR,
    apply_bandpass = True,
    apply_zscore   = True,
    target_length  = 1500,
)

---
## Stage 2 — Feature Extraction (flat per rep)

Slides a window over each rep, extracts libemg 50 features per window, concatenates all windows into one flat vector per rep.

```
window_size  = 250 samples  (~49 ms at 5120 Hz)
window_shift = 50  samples  (~10 ms stride)
windows/rep  = 26
output X     = (N_reps, 26 × n_feat_total)
```

Already-processed subjects are skipped automatically.

In [ ]:
from feature_extraction import batch_extract_features

batch_extract_features(
    input_dir    = PROC_DIR,
    output_dir   = FEAT_DIR,
    window_size  = 250,
    window_shift = 50,
)

---
## Verify Outputs

Sanity check — print shapes and feature names for one extracted file.

In [ ]:
import glob, os
import numpy as np

files = sorted(glob.glob(os.path.join(FEAT_DIR, '*_flat_rep.npz')))
print(f'{len(files)} subject file(s) found')

if files:
    d = np.load(files[0])
    print(f'X={d["X"].shape}  y={d["y"].shape}')
    print(f'windows/rep={int(d["n_windows_per_rep"])}')
    print(f'Features (first 5): {list(d["feature_list"][:5])}')
    print(f'Class dist: { {i: int((d["y"]==i).sum()) for i in range(7)} }')
    print(f'File: {os.path.basename(files[0])}')